In [5]:
##modules
#%matplotlib widget
#%matplotlib inline
#
# %matplotlib qt
import mne
import numpy as np

# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.
import matplotlib
import matplotlib.pyplot as plt

matplotlib.use('Qt5Agg')  # Asegúrate de que este backend está instalado.
mne.viz.set_browser_backend('qt')  # o 'matplotlib'

import pandas as pd 
import os
import sys

import re
from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs

from statsmodels.tsa.stattools import acf
import numpy as np

import pandas as pd
sys.path.append("..")  # esto sube un nivel desde Scripts_visual_block

from scipy.io import savemat

import pickle

Using qt as 2D backend.


In [6]:

# --- Configuración dinámica de rutas ---
from get_paths_SELF import get_paths_SELF

# Parámetros editables
disco = "g"
layer_script = "event"
subj = "s01b"

# Generar variables automáticamente
path_dict = get_paths_SELF(disco=disco, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")



✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\ICA_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_matlab_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\evoked_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_source\source_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_source\source_event\raw_hsp
✅ Carpeta creada: g:\PROYECTO_SELF\output_source\source_event\fwd
✅ Carpeta creada: g:\PROYECTO_SELF\output_source\source_event\inverse
✅ Carpeta creada: g:\PROYECTO_SELF\output_analysis\analysis_event\acw_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_analysis\analysis_event\PLE_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_analysis\analysis_event\ISC_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_analysis\analysis_block\ISC_block

📁 Rutas generadas:
datadir      

In [7]:


##adiciones en el script
number_script=3
#only for  control scripts
process="preproc"
#acuerdate de cambiar el nombre del script de auditory a block
disco="g"
# layer of script



# Definir la ruta del log
log_path = preproc_path /f"Logs_preproc_{layer_script}_{number_script}"
log_path.mkdir(parents=True, exist_ok=True)  # Crear la carpeta de logs si no existe

# Definir la ruta de logs de error
error_log_path = log_path / f"Logs_{layer_script}_{number_script}_error"
error_log_path.mkdir(parents=True, exist_ok=True)  # Crear la carpeta de logs de error si no existe

#script = Path(r"G:\apuntes_mne\code_MOUS\Scripts_{layer_script}\preproc_auditory_{layer_script}_1.py")

script = Path(f"G:\\PROYECTO_SELF\\CODE_self\\Scripts_preproc_{layer_script}\\import_edf_preproc_export_epochs_{number_script}.py")

print(f"Script path: {script}")

Script path: G:\PROYECTO_SELF\CODE_self\Scripts_preproc_event\import_edf_preproc_export_epochs_3.py


In [9]:
preproc_path

WindowsPath('g:/PROYECTO_SELF/output_preproc/preproc_event')

In [10]:
log_path

WindowsPath('g:/PROYECTO_SELF/output_preproc/preproc_event/Logs_preproc_event_3')

In [ ]:

# subj = []

# # Recorremos cada subdirectorio en la carpeta base
# for subdirectorio in general_datadir.iterdir():
#     # Comprobamos que el elemento sea un directorio y que su nombre comience con 'sub-A2'
#     if modality == "visual":
#         subject_prefix = 'sub-V1'
#     elif modality == "auditory":
#         subject_prefix = 'sub-A2'
        
#     if subdirectorio.is_dir() and subdirectorio.name.startswith(subject_prefix):
#         # Añadimos el nombre del sujeto a la lista
#         subj.append(subdirectorio.name)

# print(subj)
subj = sorted({f.name.split("_")[0].lower() for f in data_task_edf.glob("*.edf")})
print(subj)



['s01b', 's02b', 's03b', 's04b', 's05b', 's06b', 's07b', 's08b', 's09b', 's10b', 's11b', 's12b', 's13b', 's14b', 's15b', 's16b', 's17b', 's18b', 's19b', 's20b', 's21b', 's22b', 's23b', 's24b', 's25b', 's26b', 's27b', 's28b', 's29b']


In [18]:
#subjects = subj[:9]
subjects =subj
subjects


['s01b',
 's02b',
 's03b',
 's04b',
 's05b',
 's06b',
 's07b',
 's08b',
 's09b',
 's10b',
 's11b',
 's12b',
 's13b',
 's14b',
 's15b',
 's16b',
 's17b',
 's18b',
 's19b',
 's20b',
 's21b',
 's22b',
 's23b',
 's24b',
 's25b',
 's26b',
 's27b',
 's28b',
 's29b']

In [8]:
# #subjects = subj[:9]
# ##revisa que sujetos has hecho ya 
# subjects =subj[58:]
# subjects

In [19]:
script

WindowsPath('G:/PROYECTO_SELF/CODE_self/Scripts_preproc_event/import_edf_preproc_export_epochs_3.py')

In [10]:
#subjects = subj
#subjects=[subj[0], subj[10],subj[20], subj[12], subj[51], subj[70]]
#subjects= []

#subjects= subj[:50]



# for i in range(0,37 ):
#     subjects.append(subj[i])
#subjects=[subj[0]]
# Función para procesar un sujeto
def process_subject(subject):
    print(f"Starting processing {script} for {subject}...")
    result = subprocess.run(
        ["python", script, subject],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding='utf-8'  # Especificar la codificación
        )
    

    # Determinar la ruta del archivo log según el resultado
    if result.returncode == 0:
        print(f"Successfully processed {subject}")
        log_file_path = log_path / f"{subject}_{process}_{modality}_{number_script}.log"
    else:
        log_file_path = error_log_path / f"{subject}_{process}_{modality}_{number_script}_error.log"
        print(f"Error processing {subject}")

    # Guardar la salida en el archivo log correspondiente
    with open(log_file_path, "w", encoding='utf-8') as log_file:
        log_file.write(result.stdout)
        log_file.write(result.stderr)

    print(f"Log for {subject} saved to {log_file_path}")

# Crear un ThreadPoolExecutor para ejecutar el script en paralelo
with concurrent.futures.ThreadPoolExecutor(max_workers=3 ) as executor:
    # Ejecutar la función process_subject para cada sujeto
    executor.map(process_subject, subjects)


Starting processing G:\apuntes_mne\code_MOUS\Scripts_visual_block\preproc_visual_block_1.py for sub-V1001...
Starting processing G:\apuntes_mne\code_MOUS\Scripts_visual_block\preproc_visual_block_1.py for sub-V1002...
Starting processing G:\apuntes_mne\code_MOUS\Scripts_visual_block\preproc_visual_block_1.py for sub-V1003...
Error processing sub-V1003
Log for sub-V1003 saved to g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\Logs_preproc_block_1\Logs_block_1_error\sub-V1003_preproc_visual_1_error.log
Starting processing G:\apuntes_mne\code_MOUS\Scripts_visual_block\preproc_visual_block_1.py for sub-V1004...
Error processing sub-V1004
Log for sub-V1004 saved to g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\Logs_preproc_block_1\Logs_block_1_error\sub-V1004_preproc_visual_1_error.log
Starting processing G:\apuntes_mne\code_MOUS\Scripts_visual_block\preproc_visual_block_1.py for sub-V1005...
Successfully processed sub-V1001
Log for sub-V1001 saved to g:\MOUS_204\MOUS_visual\out